# 01 Data Cleaning

This notebook focuses on loading, reviewing, and cleaning the Online Retail II dataset.

The goal is to prepare reliable transaction-level and product-level datasets for SQL analysis, RFM customer segmentation, cohort analysis, and Tableau dashboard development.

Main cleaning tasks:
- Load both Excel sheets and combine them into one dataset
- Check missing values, duplicate records, cancelled orders, and abnormal price/quantity records
- Define clear revenue concepts including gross sales revenue, transaction net revenue, and customer-level revenue
- Separate non-sales records such as price anomalies, manual adjustments, postage, Amazon fees, and bad debt adjustments
- Create cleaned datasets for sales analysis, product analysis, customer analysis, and data quality review
- Export cleaned datasets and QA summaries for downstream analysis

In [1]:
import pandas as pd
import numpy as np 

file_path = "../data/raw/online_retail_II.xlsx"
xls = pd.ExcelFile(file_path)
xls.sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [2]:
df_2009_2010 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel(file_path, sheet_name='Year 2010-2011')

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

(1067371, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB
None
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [4]:
print(df.describe())
print(df.duplicated().sum())

           Quantity                 InvoiceDate         Price    Customer ID
count  1.067371e+06                     1067371  1.067371e+06  824364.000000
mean   9.938898e+00  2011-01-02 21:13:55.394029  4.649388e+00   15324.638504
min   -8.099500e+04         2009-12-01 07:45:00 -5.359436e+04   12346.000000
25%    1.000000e+00         2010-07-09 09:46:00  1.250000e+00   13975.000000
50%    3.000000e+00         2010-12-07 15:28:00  2.100000e+00   15255.000000
75%    1.000000e+01         2011-07-22 10:23:00  4.150000e+00   16797.000000
max    8.099500e+04         2011-12-09 12:50:00  3.897000e+04   18287.000000
std    1.727058e+02                         NaN  1.235531e+02    1697.464450
34335


In [5]:
#check cancelled order
df['Invoice'] = df['Invoice'].astype(str)
cancelled_orders = df[df['Invoice'].str.startswith('C')]
print(cancelled_orders.shape)
print(cancelled_orders.head())

(19494, 8)
     Invoice StockCode                    Description  Quantity  \
178  C489449     22087       PAPER BUNTING WHITE LACE       -12   
179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   
180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   
181  C489449     21896             POTTING SHED TWINE        -6   
182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   

            InvoiceDate  Price  Customer ID    Country  
178 2009-12-01 10:33:00   2.95      16321.0  Australia  
179 2009-12-01 10:33:00   1.65      16321.0  Australia  
180 2009-12-01 10:33:00   4.25      16321.0  Australia  
181 2009-12-01 10:33:00   2.10      16321.0  Australia  
182 2009-12-01 10:33:00   2.95      16321.0  Australia  


## Initial Data Quality Findings

The raw dataset contains 1,067,371 transaction-level records and 8 columns.

Initial data quality checks identified several issues:
1. 4,382 missing values in `Description`
2. 243,007 missing values in `Customer ID`
3. 34,335 duplicated rows
4. 19,494 cancelled transaction lines
5. Negative values in `Quantity`
6. Zero or negative values in `Price`

Because different business questions require different data definitions, this project creates multiple cleaned datasets instead of using one single cleaned table.

The main cleaned datasets are:
- `clean_all_transactions.csv`: transaction-level data for net revenue analysis
- `clean_positive_sales.csv`: positive sales activity before excluding non-product records
- `clean_product_sales.csv`: product-level sales data for product, country, and dashboard analysis
- `clean_customer_transactions.csv`: customer-level data for RFM, cohort, and retention analysis
- `price_anomalies.csv`: zero or negative price records for data quality review
- `non_product_transactions.csv`: manual, postage, Amazon fee, bad debt, and other non-product records

In [6]:
#Create a working copy
df_clean = df.copy()

#standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_clean = df_clean.rename(columns={
    "invoicedate": "invoice_date"
})
df_clean.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
# Convert invoice and stock code to string
df_clean['invoice'] = df_clean['invoice'].astype(str)
df_clean['stockcode'] = df_clean['stockcode'].astype(str)

# Clean text fileds
df_clean['description'] = df_clean['description'].astype("string").str.strip()
df_clean['country'] = df_clean['country'].astype("string").str.strip()

# Convert customer_id to nullable integer type
df_clean['customer_id'] = pd.to_numeric(df_clean['customer_id'], errors="coerce")

# Make sure invoice_date is datetime
df_clean['invoice_date'] = pd.to_datetime(df_clean['invoice_date'])

# Flag cancelled orders
df_clean['is_cancelled'] = df_clean['invoice'].str.startswith("C")

# Create revenue filed

df_clean['revenue'] = df_clean['quantity'] * df_clean['price']

df_clean.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,is_cancelled,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0


In [8]:
data_quality_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "missing_description",
        "missing_customer_id",
        "duplicate_rows",
        "cancelled_orders",
        "quantity_less_equal_zero",
        "price_less_equal_zero"
    ],
    "count":[
        len(df_clean),
        df_clean["description"].isna().sum(),
        df_clean["customer_id"].isna().sum(),
        df_clean.duplicated().sum(),
        df_clean['is_cancelled'].sum(),
        (df_clean['quantity'] <= 0).sum(),
        (df_clean['price'] <= 0).sum()
    ]
})

data_quality_summary['percentage'] = (
    data_quality_summary['count'] / len(df_clean) * 100
).round(2)

data_quality_summary

,metric,count,percentage
0,total_rows,1067371,100.00
1,missing_description,4382,0.41
2,missing_customer_id,243007,22.77
3,duplicate_rows,34335,3.22
4,cancelled_orders,19494,1.83
5,quantity_less_equal_zero,22950,2.15
6,price_less_equal_zero,6207,0.58


## Revenue Definition and Cleaning Strategy

The dataset contains positive sales transactions, cancelled orders, returns, price anomalies, and non-product adjustment records.

Simply removing cancelled or negative transactions may overstate actual business revenue because some negative records may offset previous positive purchases. At the same time, not every positive-price record is a normal product sale. Some positive records represent postage, manual adjustments, Amazon fees, or bad debt adjustments.

Therefore, this project separates revenue into several business definitions:

1. **Gross Sales Revenue**  
   Revenue from positive sales records before excluding non-product transactions. This reflects overall positive sales activity.

2. **Product Sales Revenue**  
   Revenue from positive product sales after excluding manual adjustments, postage, Amazon fees, bad debt adjustments, and other non-product records. This is used for product analysis, country analysis, and Tableau dashboard views.

3. **Transaction Net Revenue**  
   Revenue after basic cleaning, excluding zero or negative price anomalies but keeping both positive sales and return/cancellation records. This is used to evaluate transaction-level net revenue.

4. **Customer-Level Revenue**  
   Revenue from records with valid Customer ID. This is used for RFM segmentation, cohort analysis, repeat purchase analysis, and customer value analysis.

To avoid mixing business definitions, this notebook creates separate datasets for transaction-level analysis, product-level analysis, customer-level analysis, and data quality review.

### Price anomaly check

In [9]:
price_issue_summary = pd.DataFrame({
    "price_condition": [
        "price_greater_than_zero",
        "price_equal_zero",
        "price_less_than_zero"
    ],
    "row_count": [
        (df_clean["price"] > 0).sum(),
        (df_clean["price"] == 0).sum(),
        (df_clean["price"] < 0).sum()
    ],
    "revenue_sum": [
        df_clean.loc[df_clean["price"] > 0, "revenue"].sum(),
        df_clean.loc[df_clean["price"] == 0, "revenue"].sum(),
        df_clean.loc[df_clean["price"] < 0, "revenue"].sum()
    ],
    "unique_invoices": [
        df_clean.loc[df_clean["price"] > 0, "invoice"].nunique(),
        df_clean.loc[df_clean["price"] == 0, "invoice"].nunique(),
        df_clean.loc[df_clean["price"] < 0, "invoice"].nunique()
    ],
    "unique_customers": [
        df_clean.loc[df_clean["price"] > 0, "customer_id"].nunique(),
        df_clean.loc[df_clean["price"] == 0, "customer_id"].nunique(),
        df_clean.loc[df_clean["price"] < 0, "customer_id"].nunique()
    ]
})

price_issue_summary

,price_condition,row_count,revenue_sum,unique_invoices,unique_customers
0,price_greater_than_zero,1061164,1.944593e+07,48369,5939
1,price_equal_zero,6202,0.000000e+00,5379,51
2,price_less_than_zero,5,-1.586761e+05,5,0


In [10]:
df_clean[df_clean["price"] <= 0][
    ["invoice", "stockcode", "description", "quantity", "invoice_date", "price", "customer_id", "country", "revenue"]
].sort_values("price").head(50)

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,revenue
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,-38925.87
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
1060788,581204,85104,????damages????,-355,2011-12-07 18:32:00,0.00,NaN,United Kingdom,-0.00
1060787,581203,23406,<NA>,15,2011-12-07 18:31:00,0.00,NaN,United Kingdom,0.00
1060786,581202,23404,check,41,2011-12-07 18:30:00,0.00,NaN,United Kingdom,0.00
1060785,581201,22217,damages?,-155,2011-12-07 18:30:00,0.00,NaN,United Kingdom,-0.00
1060784,581200,84508C,check,-21,2011-12-07 18:27:00,0.00,NaN,United Kingdom,-0.00


#### Price Anomaly Interpretation

After reviewing records with zero or negative prices, most `price = 0` records appear to be non-sales operational records such as stock checks, damaged items, lost items, or inventory adjustments. These records usually have missing Customer ID, zero revenue, and are concentrated around specific dates.

Records with negative prices are mainly labeled as bad debt adjustments, which are financial adjustment records rather than normal product sales.

Therefore, this project separates `price <= 0` records into a standalone anomaly dataset. These records are excluded from sales, customer segmentation, and dashboard analysis, but retained as data quality evidence.

In [11]:
price_anomalies = df_clean[df_clean['price'] <= 0].copy()
price_anomalies.shape

(6207, 10)

In [12]:
price_anomaly_summary = (
    price_anomalies
    .assign(invoice_month=price_anomalies["invoice_date"].dt.to_period("M").astype(str))
    .groupby(["invoice_month", "description"], dropna=False)
    .agg(
        rows=("invoice", "count"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        missing_customer_id=("customer_id", lambda x: x.isna().sum())
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

price_anomaly_summary.head(20)

,invoice_month,description,rows,total_quantity,total_revenue,missing_customer_id
50,2010-02,<NA>,510,-107,0.0,510
187,2010-05,<NA>,476,-1506,0.0,476
511,2010-11,<NA>,312,-11165,0.0,312
84,2010-03,<NA>,252,-14568,0.0,252
625,2010-12,<NA>,232,-2497,0.0,232
23,2009-12,<NA>,228,8722,0.0,228
791,2011-04,<NA>,226,-3908,0.0,226
33,2010-01,<NA>,222,-10091,0.0,222
278,2010-06,<NA>,175,-3738,0.0,175
768,2011-03,<NA>,165,296,0.0,165


### Duplicate audit

In [13]:
duplicate_removed = df_clean[df_clean.duplicated(keep="first")].copy()

duplicate_audit_summary = pd.DataFrame({
    "metric": [
        "duplicate_rows_removed",
        "duplicate_revenue_sum",
        "duplicate_avg_revenue",
        "duplicate_unique_invoices",
        "duplicate_unique_customers"
    ],
    "value": [
        len(duplicate_removed),
        duplicate_removed["revenue"].sum(),
        duplicate_removed["revenue"].mean(),
        duplicate_removed["invoice"].nunique(),
        duplicate_removed["customer_id"].nunique()
    ]
})

duplicate_audit_summary

,metric,value
0,duplicate_rows_removed,34335.000000
1,duplicate_revenue_sum,431716.870000
2,duplicate_avg_revenue,12.573667
3,duplicate_unique_invoices,5391.000000
4,duplicate_unique_customers,1943.000000


In [14]:
duplicate_audit_sample = duplicate_removed[
    ["invoice", "stockcode", "description", "quantity", "invoice_date", "price", "customer_id", "country", "revenue"]
].sort_values("revenue", ascending=False).head(20)

duplicate_audit_sample

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,revenue
540478,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,13541.33
541899,537659,21623,VINTAGE UNION JACK MEMOBOARD,600,2010-12-07 16:43:00,6.38,18102.0,United Kingdom,3828.00
545332,537899,22328,ROUND SNACK BOXES SET OF 4 FRUITS,1488,2010-12-09 10:44:00,2.55,12755.0,Japan,3794.40
541901,537659,82484,WOOD BLACK BOARD ANT WHITE FINISH,600,2010-12-07 16:43:00,4.78,18102.0,United Kingdom,2868.00
541890,537657,21623,VINTAGE UNION JACK MEMOBOARD,408,2010-12-07 16:42:00,6.38,18102.0,United Kingdom,2603.04
541902,537659,22833,HALL CABINET WITH 3 DRAWERS,72,2010-12-07 16:43:00,32.69,18102.0,United Kingdom,2353.68
541896,537659,22189,CREAM HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2328.48
541897,537659,22188,BLACK HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2328.48
541888,537657,22189,CREAM HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2245.32
541889,537657,22188,BLACK HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2245.32


### Duplicate Audit Interpretation

A total of 34,335 exact duplicate rows were removed. These duplicates were identified using full-row duplication across all columns, not partial duplication by invoice or product only. The removed duplicate rows represented approximately 431,716.87 in associated revenue, with an average revenue of 12.57 per duplicate row.

Because the duplicates are exact repeated records, they are treated as likely data export artifacts rather than valid repeated customer purchases.

In [15]:
clean_all_transactions = df_clean.copy()

# Remove rows without product description
clean_all_transactions = clean_all_transactions.dropna(subset=["description"])

# Remove duplicated rows
clean_all_transactions = clean_all_transactions.drop_duplicates()

# Exclude zero or negative price records from transaction-level analysis
# These records are kept separately in price_anomalies
clean_all_transactions = clean_all_transactions[
    clean_all_transactions["price"] > 0
].copy()

# Add business logic flags
clean_all_transactions["is_negative_quantity"] = clean_all_transactions["quantity"] < 0
clean_all_transactions["is_zero_quantity"] = clean_all_transactions["quantity"] == 0
clean_all_transactions["is_positive_sale"] = (
    (clean_all_transactions["quantity"] > 0) &
    (clean_all_transactions["is_cancelled"] == False)
)

# Recalculate revenue
clean_all_transactions["revenue"] = (
    clean_all_transactions["quantity"] * clean_all_transactions["price"]
)

clean_all_transactions.shape

(1027017, 13)

In [16]:
clean_positive_sales = clean_all_transactions[
    clean_all_transactions["is_positive_sale"] == True
].copy()

clean_positive_sales.shape

(1007913, 13)

## Non-Product Transaction Handling

Outlier review showed that some high-revenue positive records are not normal product sales. Examples include manual adjustments, Amazon fees, postage, and bad debt adjustment records.

Although these records may have positive quantity and positive price, they should not be mixed with product sales because they can distort product rankings, gross sales analysis, and customer value calculations.

Therefore, this project creates:
- `non_product_transactions`: fee, postage, manual, and adjustment-related records
- `clean_product_sales`: positive product sales excluding non-product transactions

The `clean_product_sales` dataset will be used as the main dataset for product performance, country performance, and Tableau dashboard analysis.

In [17]:
non_product_stockcodes = [
    "M",
    "AMAZONFEE",
    "POST",
    "B",
    "BANK CHARGES",
    "C2",
    "DOT",
    "PADS"
]

clean_positive_sales["is_non_product_transaction"] = (
    clean_positive_sales["stockcode"].isin(non_product_stockcodes) |
    clean_positive_sales["description"].str.upper().str.contains(
        "AMAZON FEE|POSTAGE|MANUAL|ADJUST BAD DEBT|BANK CHARGES|DOTCOM POSTAGE",
        na=False
    )
)

non_product_transactions = clean_positive_sales[
    clean_positive_sales["is_non_product_transaction"] == True
].copy()

clean_product_sales = clean_positive_sales[
    clean_positive_sales["is_non_product_transaction"] == False
].copy()

non_product_summary = pd.DataFrame({
    "dataset": [
        "clean_positive_sales",
        "non_product_transactions",
        "clean_product_sales"
    ],
    "rows": [
        len(clean_positive_sales),
        len(non_product_transactions),
        len(clean_product_sales)
    ],
    "revenue": [
        clean_positive_sales["revenue"].sum(),
        non_product_transactions["revenue"].sum(),
        clean_product_sales["revenue"].sum()
    ],
    "unique_customers": [
        clean_positive_sales["customer_id"].nunique(),
        non_product_transactions["customer_id"].nunique(),
        clean_product_sales["customer_id"].nunique()
    ]
})

non_product_summary

,dataset,rows,revenue,unique_customers
0,clean_positive_sales,1007913,2.047626e+07,5878
1,non_product_transactions,4439,8.202529e+05,874
2,clean_product_sales,1003474,1.965601e+07,5861


In [18]:
product_outlier_summary = pd.DataFrame({
    "metric": [
        "quantity_99th_percentile",
        "quantity_999th_percentile",
        "quantity_max",
        "price_99th_percentile",
        "price_999th_percentile",
        "price_max",
        "revenue_99th_percentile",
        "revenue_999th_percentile",
        "revenue_max"
    ],
    "value": [
        clean_product_sales["quantity"].quantile(0.99),
        clean_product_sales["quantity"].quantile(0.999),
        clean_product_sales["quantity"].max(),
        clean_product_sales["price"].quantile(0.99),
        clean_product_sales["price"].quantile(0.999),
        clean_product_sales["price"].max(),
        clean_product_sales["revenue"].quantile(0.99),
        clean_product_sales["revenue"].quantile(0.999),
        clean_product_sales["revenue"].max()
    ]
})

product_outlier_summary

,metric,value
0,quantity_99th_percentile,108.0000
1,quantity_999th_percentile,500.0000
2,quantity_max,80995.0000
3,price_99th_percentile,16.9500
4,price_999th_percentile,35.7500
5,price_max,5117.0300
6,revenue_99th_percentile,179.0000
7,revenue_999th_percentile,764.3689
8,revenue_max,168469.6000


In [19]:
top_product_sales_outliers = clean_product_sales[
    [
        "invoice",
        "stockcode",
        "description",
        "quantity",
        "invoice_date",
        "price",
        "customer_id",
        "country",
        "revenue"
    ]
].sort_values("revenue", ascending=False).head(20)

top_product_sales_outliers

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,revenue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,38970.00
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838.0,United Kingdom,15818.40
228042,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008.0,United Kingdom,8925.00
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450.0,United Kingdom,7144.72
578172,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40
686007,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40
461644,533027,22086,PAPER CHAIN KIT 50'S CHRISTMAS,835,2010-11-15 16:02:00,6.95,NaN,United Kingdom,5803.25
379875,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,2010-10-08 10:10:00,1.66,15838.0,United Kingdom,5179.20


### Product Sales Outlier Interpretation

The `clean_product_sales` dataset contains extreme high-value product transaction lines. Since the 99.9th percentile of line revenue is much lower than the maximum line revenue, these records may represent either valid wholesale/B2B bulk purchases or potential data entry anomalies.

Instead of removing these records immediately, this project keeps them in the product sales dataset and documents them through an outlier summary. This allows the analysis to preserve possible real high-value wholesale transactions while making their potential impact on product rankings, customer value, and dashboard KPIs transparent.

In [20]:
clean_customer_transactions = clean_all_transactions.dropna(subset=["customer_id"]).copy()

clean_customer_transactions.shape

(797815, 13)

In [21]:
datasets = [
    clean_all_transactions,
    clean_positive_sales,
    clean_product_sales,
    clean_customer_transactions,
    non_product_transactions,
    price_anomalies
]

for dataset in datasets:
    dataset["invoice_year"] = dataset["invoice_date"].dt.year
    dataset["invoice_month"] = dataset["invoice_date"].dt.to_period("M").astype(str)
    dataset["invoice_day"] = dataset["invoice_date"].dt.date
    dataset["invoice_hour"] = dataset["invoice_date"].dt.hour
    dataset["day_of_week"] = dataset["invoice_date"].dt.day_name()

clean_all_transactions.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country,is_cancelled,revenue,is_negative_quantity,is_zero_quantity,is_positive_sale,invoice_year,invoice_month,invoice_day,invoice_hour,day_of_week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,False,False,True,2009,2009-12,2009-12-01,7,Tuesday
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,2009,2009-12,2009-12-01,7,Tuesday
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,False,False,True,2009,2009-12,2009-12-01,7,Tuesday
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,False,False,True,2009,2009-12,2009-12-01,7,Tuesday
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,False,False,True,2009,2009-12,2009-12-01,7,Tuesday


In [22]:
revenue_definition_summary = pd.DataFrame({
    "metric": [
        "raw_revenue",
        "transaction_net_revenue",
        "gross_sales_revenue",
        "non_product_transaction_revenue",
        "product_sales_revenue",
        "return_cancellation_revenue",
        "non_sales_adjustment_revenue",
        "customer_level_net_revenue",
        "customer_level_gross_sales_revenue",
        "customer_level_product_sales_revenue"
    ],
    "value": [
        df_clean["revenue"].sum(),
        clean_all_transactions["revenue"].sum(),
        clean_positive_sales["revenue"].sum(),
        non_product_transactions["revenue"].sum(),
        clean_product_sales["revenue"].sum(),
        clean_all_transactions.loc[
            clean_all_transactions["quantity"] < 0, "revenue"
        ].sum(),
        price_anomalies["revenue"].sum(),
        clean_customer_transactions["revenue"].sum(),
        clean_customer_transactions.loc[
            clean_customer_transactions["is_positive_sale"] == True, "revenue"
        ].sum(),
        clean_product_sales.dropna(subset=["customer_id"])["revenue"].sum()
    ]
})

revenue_definition_summary

,metric,value
0,raw_revenue,1.928725e+07
1,transaction_net_revenue,1.901421e+07
2,gross_sales_revenue,2.047626e+07
3,non_product_transaction_revenue,8.202529e+05
4,product_sales_revenue,1.965601e+07
5,return_cancellation_revenue,-1.462424e+06
6,non_sales_adjustment_revenue,-1.586761e+05
7,customer_level_net_revenue,1.628999e+07
8,customer_level_gross_sales_revenue,1.737480e+07
9,customer_level_product_sales_revenue,1.707348e+07


In [23]:
cleaning_summary = pd.DataFrame({
    "dataset": [
        "raw_data",
        "clean_all_transactions",
        "clean_positive_sales",
        "non_product_transactions",
        "clean_product_sales",
        "clean_customer_transactions",
        "price_anomalies"
    ],
    "business_use": [
        "Original dataset before cleaning",
        "Transaction-level net revenue analysis including sales and returns/cancellations",
        "Positive sales activity before excluding non-product records",
        "Manual, postage, Amazon fee, bad debt, and other non-product records",
        "Product-level sales data for product, country, and dashboard analysis",
        "Customer-level transaction data for net customer value analysis; RFM and cohort datasets will be derived later",
        "Zero or negative price records for data quality review"
    ],
    "rows": [
        len(df_clean),
        len(clean_all_transactions),
        len(clean_positive_sales),
        len(non_product_transactions),
        len(clean_product_sales),
        len(clean_customer_transactions),
        len(price_anomalies)
    ],
    "unique_invoices": [
        df_clean["invoice"].nunique(),
        clean_all_transactions["invoice"].nunique(),
        clean_positive_sales["invoice"].nunique(),
        non_product_transactions["invoice"].nunique(),
        clean_product_sales["invoice"].nunique(),
        clean_customer_transactions["invoice"].nunique(),
        price_anomalies["invoice"].nunique()
    ],
    "unique_customers": [
        df_clean["customer_id"].nunique(),
        clean_all_transactions["customer_id"].nunique(),
        clean_positive_sales["customer_id"].nunique(),
        non_product_transactions["customer_id"].nunique(),
        clean_product_sales["customer_id"].nunique(),
        clean_customer_transactions["customer_id"].nunique(),
        price_anomalies["customer_id"].nunique()
    ],
    "total_revenue": [
        df_clean["revenue"].sum(),
        clean_all_transactions["revenue"].sum(),
        clean_positive_sales["revenue"].sum(),
        non_product_transactions["revenue"].sum(),
        clean_product_sales["revenue"].sum(),
        clean_customer_transactions["revenue"].sum(),
        price_anomalies["revenue"].sum()
    ]
})

cleaning_summary

,dataset,business_use,rows,unique_invoices,unique_customers,total_revenue
0,raw_data,Original dataset before cleaning,1067371,53628,5942,1.928725e+07
1,clean_all_transactions,Transaction-level net revenue analysis includi...,1027017,48369,5939,1.901421e+07
2,clean_positive_sales,Positive sales activity before excluding non-p...,1007913,40077,5878,2.047626e+07
3,non_product_transactions,"Manual, postage, Amazon fee, bad debt, and oth...",4439,4350,874,8.202529e+05
4,clean_product_sales,"Product-level sales data for product, country,...",1003474,39573,5861,1.965601e+07
5,clean_customer_transactions,Customer-level transaction data for net custom...,797815,44870,5939,1.628999e+07
6,price_anomalies,Zero or negative price records for data qualit...,6207,5384,51,-1.586761e+05


In [24]:
revenue_definition_summary_display = revenue_definition_summary.copy()
revenue_definition_summary_display["value_formatted"] = revenue_definition_summary_display["value"].apply(lambda x: f"{x:,.2f}")

revenue_definition_summary_display

,metric,value,value_formatted
0,raw_revenue,1.928725e+07,"19,287,250.57"
1,transaction_net_revenue,1.901421e+07,"19,014,209.84"
2,gross_sales_revenue,2.047626e+07,"20,476,260.45"
3,non_product_transaction_revenue,8.202529e+05,"820,252.94"
4,product_sales_revenue,1.965601e+07,"19,656,007.51"
5,return_cancellation_revenue,-1.462424e+06,"-1,462,424.18"
6,non_sales_adjustment_revenue,-1.586761e+05,"-158,676.14"
7,customer_level_net_revenue,1.628999e+07,"16,289,991.29"
8,customer_level_gross_sales_revenue,1.737480e+07,"17,374,804.27"
9,customer_level_product_sales_revenue,1.707348e+07,"17,073,476.18"


In [25]:
missing_description_records = df_clean[df_clean["description"].isna()].copy()

missing_description_summary = pd.DataFrame({
    "metric": [
        "missing_description_rows",
        "missing_description_revenue",
        "missing_description_unique_invoices",
        "missing_description_unique_customers",
        "missing_description_with_customer_id"
    ],
    "value": [
        len(missing_description_records),
        missing_description_records["revenue"].sum(),
        missing_description_records["invoice"].nunique(),
        missing_description_records["customer_id"].nunique(),
        missing_description_records["customer_id"].notna().sum()
    ]
})

missing_description_summary

,metric,value
0,missing_description_rows,4382.0
1,missing_description_revenue,0.0
2,missing_description_unique_invoices,4275.0
3,missing_description_unique_customers,0.0
4,missing_description_with_customer_id,0.0


### Missing Description Interpretation

The dataset contains 4,382 records with missing product descriptions. These records have zero total revenue and no valid Customer ID. Because they do not contribute to revenue analysis and cannot be used for customer-level analysis, they are excluded from the main cleaned datasets.

## Data Cleaning Summary

The raw dataset contains 1,067,371 transaction records. Initial data quality checks identified missing product descriptions, missing customer IDs, duplicated rows, cancelled orders, negative quantities, zero prices, negative prices, and non-product transaction records.

Instead of removing all abnormal records directly, this project applied a business-rule-based cleaning strategy:

- Positive sales transactions were separated to measure gross positive sales activity.
- Return and cancellation records were retained in the transaction-level dataset to calculate transaction net revenue.
- Zero or negative price records were reviewed and separated as price anomalies, including inventory adjustments, damaged or lost items, stock checks, and bad debt adjustments.
- Non-product positive records such as manual adjustments, postage, Amazon fees, and bad debt adjustments were separated from normal product sales.
- A clean product sales dataset was created for product performance, country performance, and dashboard analysis.
- Customer-level records with valid Customer ID were separated for customer-level net revenue analysis. RFM, cohort, and retention datasets will be derived from the appropriate cleaned datasets in later notebooks.

This approach avoids mixing different revenue definitions and makes the analysis more suitable for real business decision-making.

In [26]:
customer_id_coverage_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "rows_missing_customer_id",
        "missing_customer_id_rate",
        "transaction_net_revenue",
        "customer_level_net_revenue",
        "revenue_excluded_from_customer_analysis",
        "excluded_revenue_rate"
    ],
    "value": [
        len(df_clean),
        df_clean["customer_id"].isna().sum(),
        df_clean["customer_id"].isna().sum() / len(df_clean),
        clean_all_transactions["revenue"].sum(),
        clean_customer_transactions["revenue"].sum(),
        clean_all_transactions["revenue"].sum() - clean_customer_transactions["revenue"].sum(),
        (clean_all_transactions["revenue"].sum() - clean_customer_transactions["revenue"].sum()) / clean_all_transactions["revenue"].sum()
    ]
})

customer_id_coverage_summary

,metric,value
0,raw_rows,1.067371e+06
1,rows_missing_customer_id,2.430070e+05
2,missing_customer_id_rate,2.276687e-01
3,transaction_net_revenue,1.901421e+07
4,customer_level_net_revenue,1.628999e+07
5,revenue_excluded_from_customer_analysis,2.724219e+06
6,excluded_revenue_rate,1.432728e-01


### Customer ID Coverage Limitation

Customer-level analysis is limited to records with valid Customer ID. Around 22.77% of raw records do not contain Customer ID, which means RFM segmentation, cohort retention analysis, and customer value analysis only represent identified customers.

The customer-level dataset captures approximately 16.29M in net revenue, while the transaction-level net revenue is approximately 19.01M. Therefore, around 2.72M, or 14.33%, of transaction net revenue is excluded from customer-level analysis.

In [27]:
clean_all_transactions.to_csv("../data/processed/clean_all_transactions.csv", index=False)
clean_positive_sales.to_csv("../data/processed/clean_positive_sales.csv", index=False)
clean_product_sales.to_csv("../data/processed/clean_product_sales.csv", index=False)
clean_customer_transactions.to_csv("../data/processed/clean_customer_transactions.csv", index=False)

price_anomalies.to_csv("../data/processed/price_anomalies.csv", index=False)
non_product_transactions.to_csv("../data/processed/non_product_transactions.csv", index=False)

data_quality_summary.to_csv("../data/processed/data_quality_summary.csv", index=False)
price_issue_summary.to_csv("../data/processed/price_issue_summary.csv", index=False)
price_anomaly_summary.to_csv("../data/processed/price_anomaly_summary.csv", index=False)
duplicate_audit_summary.to_csv("../data/processed/duplicate_audit_summary.csv", index=False)
duplicate_audit_sample.to_csv("../data/processed/duplicate_audit_sample.csv", index=False)
product_outlier_summary.to_csv("../data/processed/product_outlier_summary.csv", index=False)
top_product_sales_outliers.to_csv("../data/processed/top_product_sales_outliers.csv", index=False)
non_product_summary.to_csv("../data/processed/non_product_summary.csv", index=False)
missing_description_summary.to_csv("../data/processed/missing_description_summary.csv", index=False)
customer_id_coverage_summary.to_csv("../data/processed/customer_id_coverage_summary.csv", index=False)
revenue_definition_summary.to_csv("../data/processed/revenue_definition_summary.csv", index=False)
cleaning_summary.to_csv("../data/processed/cleaning_summary.csv", index=False)

print("Cleaned datasets and QA summaries exported successfully.")

Cleaned datasets and QA summaries exported successfully.
